This notebook will be used to implement Modality-Attentive Meta-Embedding Generator and integrate it into a single unified pipeline that handles warm-start and cold-start items without separate models.

In [112]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [113]:
import os
import json
import numpy as np
import torch
import pandas as pd

In [114]:
# Base paths (your standard structure)
base_dir = "/content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M"
preprocessed_dir = os.path.join(base_dir, "AntM2C_preprocessed")
fusion_artifacts_dir = os.path.join(base_dir, "fusion_artifacts")

# Key artifact paths
fusion_config_path = os.path.join(fusion_artifacts_dir, "fusion_config.json")

df_id_encoded_path = os.path.join(preprocessed_dir, "split_features", "df_id_encoded.pkl")

text_embeddings_path = os.path.join(preprocessed_dir, "text_embeddings.npy")
time_features_path = os.path.join(preprocessed_dir, "time_features_final.pt")

time_feature_cols_path = os.path.join(preprocessed_dir, "time_feature_columns.json")

image_embeddings_path = os.path.join(base_dir, "image_embeddings_vit_base_aligned.pt")
image_missing_mask_path = os.path.join(base_dir, "image_missing_mask.pt")

print("base_dir:", base_dir)
print("preprocessed_dir:", preprocessed_dir)
print("fusion_artifacts_dir:", fusion_artifacts_dir)

base_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M
preprocessed_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed
fusion_artifacts_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/fusion_artifacts


In [115]:
# Quick existence checks
paths_to_check = {
    "fusion_config.json": fusion_config_path,
    "df_id_encoded.pkl": df_id_encoded_path,
    "text_embeddings.npy": text_embeddings_path,
    "time_features_final.pt": time_features_path,
    "time_feature_columns.json": time_feature_cols_path,
    "image_embeddings_vit_base_aligned.pt": image_embeddings_path,
    "image_missing_mask.pt": image_missing_mask_path
}

for name, p in paths_to_check.items():
    print(f"{name} exists? -> {os.path.exists(p)} | {p}")

fusion_config.json exists? -> True | /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/fusion_artifacts/fusion_config.json
df_id_encoded.pkl exists? -> True | /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed/split_features/df_id_encoded.pkl
text_embeddings.npy exists? -> True | /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed/text_embeddings.npy
time_features_final.pt exists? -> True | /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed/time_features_final.pt
time_feature_columns.json exists? -> True | /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed/time_feature_columns.json
image_embeddings_vit_base_aligned.pt exists? -> True | /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/image_embeddings_vit_base_aligned.pt
image_missing_mask.pt exists? ->

In [116]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cpu


In [117]:
# Load fusion config + vocab sizes
with open(fusion_config_path, "r") as f:
    fusion_config = json.load(f)

print("--- Loaded fusion_config.json ---")
for k in ["num_users", "num_items", "num_scenes", "text_dim", "image_dim", "time_dim", "d_model", "num_experts", "hidden_dim"]:
    if k in fusion_config:
        print(f"{k}: {fusion_config[k]}")

--- Loaded fusion_config.json ---
num_users: 67615
num_items: 59142
num_scenes: 5
text_dim: 1536
image_dim: 768
time_dim: 5
d_model: 256
num_experts: 4
hidden_dim: 256


In [118]:
# Load df_id_encoded + validate columns
df_id = pd.read_pickle(df_id_encoded_path)
print("Loaded df_id_encoded.pkl:", df_id.shape)
print("Columns:", list(df_id.columns))

required_cols = ["user_id_idx", "item_id_idx"]
scene_candidates = ["scene_idx", "scene_id_idx"]

assert all(c in df_id.columns for c in required_cols), f"Missing required cols. Expected {required_cols}"
scene_col = next((c for c in scene_candidates if c in df_id.columns), None)
assert scene_col is not None, f"Missing scene column. Expected one of {scene_candidates}"

print("Detected scene column:", scene_col)

df_id.head()

Loaded df_id_encoded.pkl: (3730329, 3)
Columns: ['user_id_idx', 'item_id_idx', 'scene_idx']
Detected scene column: scene_idx


,user_id_idx,item_id_idx,scene_idx
0,41688,13590,4
1,39834,21351,2
2,22354,33398,2
3,17547,11190,1
4,55873,33398,2


In [119]:
# Load embeddings/features
# Text embeddings: memmap (interaction-level)
text_embeddings = np.load(text_embeddings_path, mmap_mode="r")
num_text_rows, text_dim = text_embeddings.shape

# Time features: torch tensor (interaction-level)
time_features = torch.load(time_features_path, map_location="cpu")
num_time_rows, time_dim = time_features.shape

# Time feature columns
with open(time_feature_cols_path, "r") as f:
    time_feature_cols = json.load(f)

# Image embeddings: aligned item-level
image_embeddings = torch.load(image_embeddings_path, map_location="cpu")
num_items_img, image_dim = image_embeddings.shape

# Missing mask
image_missing_mask = torch.load(image_missing_mask_path, map_location="cpu")
missing_count = int(image_missing_mask.sum().item())

In [120]:
# print shapes (lightweight)
print("------ Loaded Artifact Shapes ------")
print(f"df_id rows:        {len(df_id)}")
print(f"text_embeddings:   ({num_text_rows}, {text_dim})   [interaction-level]")
print(f"time_features:     ({num_time_rows}, {time_dim})   [interaction-level]")
print(f"image_embeddings:  ({num_items_img}, {image_dim})  [item-level aligned]")
print(f"missing images:    {missing_count}")
print("Time feature columns:", time_feature_cols)

------ Loaded Artifact Shapes ------
df_id rows:        3730329
text_embeddings:   (3730329, 1536)   [interaction-level]
time_features:     (3730329, 5)   [interaction-level]
image_embeddings:  (59142, 768)  [item-level aligned]
missing images:    2
Time feature columns: ['unix_time_norm', 'hour_sin', 'hour_cos', 'dayofweek', 'is_weekend']


In [121]:
# Alignment assert checks (should pass to proceed)
assert len(df_id) == num_text_rows, (
    f"Mismatch: df_id rows ({len(df_id)}) != text_embeddings rows ({num_text_rows})"
)
assert len(df_id) == num_time_rows, (
    f"Mismatch: df_id rows ({len(df_id)}) != time_features rows ({num_time_rows})"
)
assert len(time_feature_cols) == time_dim, (
    f"Mismatch: time_feature_cols ({len(time_feature_cols)}) != time_dim ({time_dim})"
)

assert num_items_img == fusion_config["num_items"], (
    f"Mismatch: image embedding rows ({num_items_img}) != num_items in fusion_config ({fusion_config['num_items']})"
)

print("Setup complete: all artifacts loaded and shapes are aligned.")

Setup complete: all artifacts loaded and shapes are aligned.


### Define Cold/Warm Conditions (Unified Logic)

frequency-based cold-start definition (few-shot) aligned with the AntM2C-style baseline idea, and computes item and user frequency counts on train only
*   few-shot item: 0 < item_train_count < N
*   few-shot user: 0 < user_train_count < N
*   cold_mask    : few-shot item OR few-shot user

Additionally, missing_image_mask is kept separately for robustness

In [122]:
splits_dir = os.path.join(preprocessed_dir, "splits")
os.makedirs(splits_dir, exist_ok=True)

train_idx_path = os.path.join(splits_dir, "train_idx.npy")
val_idx_path   = os.path.join(splits_dir, "val_idx.npy")
test_idx_path  = os.path.join(splits_dir, "test_idx.npy")

N = len(df_id)

In [145]:
if os.path.exists(train_idx_path) and os.path.exists(val_idx_path) and os.path.exists(test_idx_path):
    train_idx = np.load(train_idx_path)
    val_idx   = np.load(val_idx_path)
    test_idx  = np.load(test_idx_path)
    print("Loaded existing splits.")
else:
    rng = np.random.default_rng(42)
    perm = rng.permutation(N)

    train_end = int(0.8 * N)
    val_end   = int(0.9 * N)

    train_idx = perm[:train_end]
    val_idx   = perm[train_end:val_end]
    test_idx  = perm[val_end:]

    np.save(train_idx_path, train_idx)
    np.save(val_idx_path, val_idx)
    np.save(test_idx_path, test_idx)

    print("Created and saved new splits.")

print("Split sizes (train/validation/test):", len(train_idx), len(val_idx), len(test_idx))

Loaded existing splits.
Split sizes (train/validation/test): 2984263 373033 373033


In [146]:
# Compute train-only frequency counts (fast, vectorized) using np.bincount which is very fast for integer indices
# Train-only arrays
train_items = df_id.iloc[train_idx]["item_id_idx"].to_numpy(dtype=np.int64)
train_users = df_id.iloc[train_idx]["user_id_idx"].to_numpy(dtype=np.int64)

num_items = fusion_config["num_items"]
num_users = fusion_config["num_users"]

# bincount produces length = max_id+1, so set minlength for full vocab
item_train_count = np.bincount(train_items, minlength=num_items)
user_train_count = np.bincount(train_users, minlength=num_users)

print("item_train_count shape:", item_train_count.shape, "min/max:", item_train_count.min(), item_train_count.max())
print("user_train_count shape:", user_train_count.shape, "min/max:", user_train_count.min(), user_train_count.max())

item_train_count shape: (59142,) min/max: 0 401813
user_train_count shape: (67615,) min/max: 0 3853


Define few-shot thresholds (N=100 recommended as per AntM2C baseline-style)
* We can later tune FEWSHOT_N to (e.g., 20/50/100).

In [139]:
FEWSHOT_N = 10

# Few-shot items (TRAIN exposure)
item_is_zeroshot = (item_train_count == 0)
item_is_fewshot = (item_train_count > 0) & (item_train_count < FEWSHOT_N)
item_is_warm = (item_train_count >= FEWSHOT_N)
item_is_cold = item_is_zeroshot | item_is_fewshot


print("---- Describing item-level coverage ----")
print("Zero-shot items:", int(item_is_zeroshot.sum()), "/", num_items)
print("Few-shot items:", int(item_is_fewshot.sum()), "/", num_items)
print("Warm items:", int(item_is_warm.sum()), "/", num_items)

# Keeps user stats for reporting only (NOT used in gating/blending)
user_is_fewshot = (user_train_count > 0) & (user_train_count < FEWSHOT_N)
user_is_warm = (user_train_count >= FEWSHOT_N)
print("Few-shot users (reporting only):", int(user_is_fewshot.sum()), "/", num_users)

---- Describing item-level coverage ----
Zero-shot items: 4493 / 59142
Few-shot items: 39751 / 59142
Warm items: 14898 / 59142
Few-shot users (reporting only): 3871 / 67615


##### 1) “Few-shot items: 39751 / 59142”
Out of 59,142 unique items in the vocabulary:
* 39,751 items appear in the training split between 1 and 9 times
(because 0 < count < 10)
##### 2) “Warm items: 14898 / 59142”
* 14,898 items appear 10 or more times in training
(count >= 10)

##### 3) Zero Shot: 4493
* 39,751 (few-shot) + 14,898 (warm) = 54,649
* 59,142 (all) − 54,649 = 4,493 items have train count = 0
* These items do not appear in training at all. (zero-shot)

Note that we have three groups:
- Zero-shot  : 4,493 items
- Few-shot   : 39,751 items
- Warm (≥10) : 14,898 items

Build cold_mask per batch
* This functoin produces a boolean mask (B,) for any item_idx_tensor / user_idx_tensor.

In [126]:
def cold_mask_fewshot(
    user_idx_tensor: torch.Tensor,
    item_idx_tensor: torch.Tensor,
    mode: str = "item_or_user"
) -> torch.Tensor:
    """
    Returns (B,) boolean mask where True indicates cold-start (few-shot) based on TRAIN counts.

    mode:
      - "item_only": cold if item is few-shot
      - "user_only": cold if user is few-shot
      - "item_or_user": cold if item OR user is few-shot (recommended)
    """
    u_cpu = user_idx_tensor.detach().cpu().numpy().astype(np.int64)
    i_cpu = item_idx_tensor.detach().cpu().numpy().astype(np.int64)

    item_cold = item_is_fewshot[i_cpu]
    user_cold = user_is_fewshot[u_cpu]

    if mode == "item_only":
        mask = item_cold
    elif mode == "user_only":
        mask = user_cold
    elif mode == "item_or_user":
        mask = item_cold | user_cold
    else:
        raise ValueError(f"Unknown mode: {mode}")

    return torch.tensor(mask, dtype=torch.bool, device=item_idx_tensor.device)

In [140]:
def cold_mask_item(item_idx_tensor: torch.Tensor) -> torch.Tensor:
    i_cpu = item_idx_tensor.detach().cpu().numpy().astype(np.int64)
    return torch.tensor(item_is_cold[i_cpu], dtype=torch.bool, device=item_idx_tensor.device)

In [131]:
def cold_mask_fewshot_item_only(item_idx_tensor: torch.Tensor) -> torch.Tensor:
    """
    Returns (B,) boolean mask where True indicates cold-start due to FEW-SHOT ITEM:
      0 < item_train_count < FEWSHOT_N
    """
    i_cpu = item_idx_tensor.detach().cpu().numpy().astype(np.int64)
    mask = item_is_fewshot[i_cpu]
    return torch.tensor(mask, dtype=torch.bool, device=item_idx_tensor.device)

# Missing-modality flag (robustness only; not used as cold-start benchmark)
def missing_image_flag(item_idx_tensor: torch.Tensor) -> torch.Tensor:
    i_cpu = item_idx_tensor.detach().cpu()
    return image_missing_mask[i_cpu].to(item_idx_tensor.device)

In [127]:
# Additional Missing-modality flag (robustness, NOT a cold-start definition)
def missing_image_flag(item_idx_tensor: torch.Tensor) -> torch.Tensor:
    i_cpu = item_idx_tensor.detach().cpu()
    return image_missing_mask[i_cpu].to(item_idx_tensor.device)

In [138]:
# Sample a small batch from TRAIN rows
sample_rows = np.random.default_rng(42).choice(train_idx, size=4096, replace=False)
sample_df = df_id.iloc[sample_rows]

it = torch.tensor(sample_df["item_id_idx"].to_numpy(), dtype=torch.long, device=device)

cold = cold_mask_fewshot_item_only(it)
miss_img = missing_image_flag(it)

print("---- Interaction-level coverage ----")
print("Estimated item-only few-shot cold rate (train sample):", float(cold.float().mean().cpu()))
print("Missing-image rate (train sample):", float(miss_img.float().mean().cpu()), "(expected ~0)")
print("Missing-image count in sample:", int(miss_img.sum().cpu()))

---- Interaction-level coverage ----
Estimated item-only few-shot cold rate (train sample): 0.033935546875
Missing-image rate (train sample): 0.0 (expected ~0)
Missing-image count in sample: 0


In [151]:
# Interaction-level diagnostic on TRAIN split only (Reporting/Understanding purposes)
train_df = df_id.iloc[train_idx]
train_item_idx = train_df["item_id_idx"].to_numpy(dtype=np.int64)

# Interaction masks
zeroshot_mask = item_is_zeroshot[train_item_idx]
fewshot_mask  = item_is_fewshot[train_item_idx]
warm_mask     = item_is_warm[train_item_idx]

num_train_interactions = len(train_item_idx)

zeroshot_interactions = zeroshot_mask.sum()
fewshot_interactions  = fewshot_mask.sum()
warm_interactions     = warm_mask.sum()

In [153]:
print("--- Interaction-level Cold-Start Breakdown (TRAIN) ---")
print(f"Total train interactions: {num_train_interactions:,}")
print()
print(f"Zero-shot interactions: {zeroshot_interactions:,} "
      f"({zeroshot_interactions / num_train_interactions:.4%})")
print(f"Few-shot interactions:  {fewshot_interactions:,} "
      f"({fewshot_interactions / num_train_interactions:.4%})")
print(f"Warm interactions:      {warm_interactions:,} "
      f"({warm_interactions / num_train_interactions:.4%})")

print("\nSanity check (should sum to 100%):",
      (zeroshot_interactions + fewshot_interactions + warm_interactions) == num_train_interactions)

--- Interaction-level Cold-Start Breakdown (TRAIN) ---
Total train interactions: 2,984,263

Zero-shot interactions: 0 (0.0000%)
Few-shot interactions:  100,189 (3.3572%)
Warm interactions:      2,884,074 (96.6428%)

Sanity check (should sum to 100%): True


- Zero-shot interactions = 0 in train is expected by definition.
If an item appears in the training interactions, its train count cannot be 0. Zero-shot items only become visible when we evaluate on validation/test or on a held-out-item split.
- Few-shot interactions = 100,189 (3.36%) is a strong, realistic cold-start slice. This is large enough for meaningful learning and evaluation, and small enough to remain a distinct “cold” regime.
- Warm interactions dominate (96.64%) which is typical in long-tail CTR datasets.

In [154]:
!

checkpoints/  explainability/  notebooks/        results/
datasets/     models/          requirements.txt  src/
